# Tornado Experiment — Master Runner

Executes the full analysis pipeline without visualizations:

1. **Load run data** — attribute tables + wide-format simulation output
2. **Post-processing** — intertemporal decomposition / rescaling
3. **Cost-benefits** — system + technical costs → `cost_benefits_data_tornado.csv`
4. **MAC analysis** — marginal abatement costs → `marginal_abatement_costs_tornado.csv`
5. **Tableau export** — tornado plot data → `Tableau/data/tableau_tornado.csv`

All configuration lives in `scripts/config.py`.  
Shared pipeline modules (`data_loading`, `postprocessing`, `cost_benefits_pipeline`) are imported from `../shared_scripts/`.


## 0 · Environment setup

In [1]:
import os
import sys
import pathlib
import logging
import warnings

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
RUNNER_DIR  = pathlib.Path(os.getcwd()).resolve()
SCRIPTS_DIR = RUNNER_DIR / "scripts"
SHARED_DIR  = RUNNER_DIR.parent / "shared_scripts"

assert SCRIPTS_DIR.exists(), f"scripts/ not found at {SCRIPTS_DIR}"
assert SHARED_DIR.exists(),  f"shared_scripts/ not found at {SHARED_DIR}"

for p in (SCRIPTS_DIR, SHARED_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ── Config ────────────────────────────────────────────────────────────────────
import config as cfg

# Project root (needed for ssp_modeling.* imports inside pipeline scripts)
if str(cfg.PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.PROJECT_DIR))

# notebooks/ dir (needed for utils.logger_utils)
if str(cfg.NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.NOTEBOOKS_DIR))

from utils.logger_utils import setup_clean_logger, mute_external_loggers

logger = setup_clean_logger("tornado", logging.INFO)
mute_external_loggers(["sisepuede"])
logger.info("Environment ready.")
logger.info(f"Project dir : {cfg.PROJECT_DIR}")
logger.info(f"Run output  : {cfg.RUN_ID_OUTPUT_DIR}")

2026-03-27 18:13:37,991 - INFO - Environment ready.
2026-03-27 18:13:37,992 - INFO - Project dir : /Users/fabianfuentes/git/ssp_uganda_data
2026-03-27 18:13:37,992 - INFO - Run output  : /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/ssp_run_output/sisepuede_run_2026-03-10t13;27;53.264959


## 1 · Load run data

In [2]:
from data_loading import load_attribute_tables, parse_strategy_metadata, load_wide_export

# Attribute tables
att_primary, att_strategy = load_attribute_tables(cfg.RUN_ID_OUTPUT_DIR)
att_strategy = parse_strategy_metadata(att_strategy)

logger.info(f"att_primary  : {att_primary.shape}")
logger.info(f"att_strategy : {att_strategy.shape}")

# Wide-format simulation output
df_export = load_wide_export(cfg.RUN_ID_OUTPUT_DIR, cfg.PRIMARY_IDS_FILTER)
logger.info(f"df_export    : {df_export.shape}")

2026-03-27 18:13:38,260 - INFO - att_primary  : (122, 4)
2026-03-27 18:13:38,260 - INFO - att_strategy : (194, 10)
2026-03-27 18:13:39,568 - INFO - df_export    : (3304, 4054)


## 2 · Post-processing — intertemporal decomposition

In [3]:
from postprocessing import run_decomposition

df_decomposed = run_decomposition(
    df_export    = df_export,
    project_dir  = cfg.PROJECT_DIR,
    targets_path = cfg.TARGETS_PATH,
    iso_code3    = cfg.ISO_CODE3,
    year_ref     = cfg.YEAR_REF,
    region       = cfg.REGION,
    output_path  = cfg.OUTPUT_DECOMPOSED,
)
logger.info(f"df_decomposed : {df_decomposed.shape}  → {cfg.OUTPUT_DECOMPOSED}")

Changed 59 zero(s) in: emission_co2e_co2_lndu_conversion_croplands_to_croplands (time_period == 4)
Changed 59 zero(s) in: emission_co2e_co2_lndu_conversion_grasslands_to_grasslands (time_period == 4)
Changed 59 zero(s) in: emission_co2e_co2_lndu_conversion_grasslands_to_pastures (time_period == 4)
Changed 59 zero(s) in: emission_co2e_co2_lndu_conversion_pastures_to_grasslands (time_period == 4)
Changed 59 zero(s) in: emission_co2e_co2_lndu_conversion_pastures_to_pastures (time_period == 4)
Changed 59 zero(s) in: emission_co2e_co2_lndu_biomass_sequestration_grasslands (time_period == 4)
Changed 59 zero(s) in: emission_co2e_co2_lndu_biomass_sequestration_pastures (time_period == 4)
Changed 59 zero(s) in: emission_co2e_n2o_lsmm_indirect_composting (time_period == 4)
Changed 59 zero(s) in: emission_co2e_n2o_lsmm_indirect_deep_bedding (time_period == 4)
Changed 59 zero(s) in: emission_co2e_n2o_lsmm_indirect_incineration (time_period == 4)
Changed 59 zero(s) in: emission_co2e_n2o_lsmm_indire

2026-03-27 18:13:53,464 - INFO - df_decomposed : (3068, 4054)  → /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/ssp_run_output/sisepuede_run_2026-03-10t13;27;53.264959/decomposed_ssp_output_tornado.csv


Decomposed output written to: /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/ssp_run_output/sisepuede_run_2026-03-10t13;27;53.264959/decomposed_ssp_output_tornado.csv
Done: uganda


## 3 · Cost-benefits

In [4]:
from cost_benefits_pipeline import run_cost_benefits

cb_data = run_cost_benefits(
    df_decomposed      = df_decomposed,
    att_primary        = att_primary,
    att_strategy       = att_strategy,
    cb_config_path     = cfg.CB_CONFIG_PATH,
    run_output_dir     = cfg.RUN_ID_OUTPUT_DIR,
    project_dir        = cfg.PROJECT_DIR,
    strategy_code_base = cfg.STRATEGY_CODE_BASE,
)
logger.info(f"cb_data : {cb_data.shape}  → {cfg.OUTPUT_CB_DATA}")

The TX TX:FRST:INCREASE_SEQUESTRATION_NZ is missing on AttTransformationCode
The TX TX:LNDU:DEC_WETLAND_LOSS_NZ is missing on AttTransformationCode
The TX TX:LNDU:SET_WETLANDS_MINIMUM_NZ is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT_NZ is missing on AttTransformationCode
The TX TX:FRST:INCREASE_SEQUESTRATION_NDC_2 is missing on AttTransformationCode
The TX TX:LNDU:DEC_WETLAND_LOSS_NDC_2 is missing on AttTransformationCode
The TX TX:LNDU:SET_WETLANDS_MINIMUM_NDC_2 is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT_NDC_2 is missing on AttTransformationCode
The TX TX:FRST:INCREASE_SEQUESTRATION_NDC_25 is missing on AttTransformationCode
The TX TX:LNDU:DEC_WETLAND_LOSS_NDC_25 is missing on AttTransformationCode
The TX TX:LNDU:SET_WETLANDS_MINIMUM_NDC_25 is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT_NDC_25 is missing on AttTransformationCode
The TX TX:FRST:INCREASE_SEQUESTRATION_NZ is missing on AttTransformationCode
The 

2026-03-27 18:21:48,056 - INFO - cb_data : (505820, 21)  → /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/ssp_run_output/sisepuede_run_2026-03-10t13;27;53.264959/cost_benefits_data_tornado.csv


## 4 · Marginal Abatement Cost (MAC)

In [5]:
from mac_pipeline import run_mac_analysis

mac_df = run_mac_analysis(
    df_decomposed           = df_decomposed,
    cb_data                 = cb_data,
    att_primary             = att_primary,
    att_strategy            = att_strategy,
    iso_code3               = cfg.ISO_CODE3,
    region                  = cfg.REGION,
    invent_dir              = cfg.INVENT_DIR,
    run_output_dir          = cfg.RUN_ID_OUTPUT_DIR,
    strategy_code_baseline  = cfg.STRATEGY_CODE_BASELINE,
)
logger.info(f"mac_df : {mac_df.shape}  → {cfg.OUTPUT_MAC}")

2026-03-27 18:21:48,543 - INFO - mac_df : (44, 9)  → /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/ssp_run_output/sisepuede_run_2026-03-10t13;27;53.264959/marginal_abatement_costs_tornado.csv


## 5 · Tableau export

In [6]:
from mac_pipeline import build_tableau_tornado

tableau_tornado = build_tableau_tornado(
    mac_df         = mac_df,
    run_output_dir = cfg.RUN_ID_OUTPUT_DIR,
    tableau_dir    = cfg.TABLEAU_DIR,
)
logger.info(f"tableau_tornado : {tableau_tornado.shape}  → {cfg.OUTPUT_TABLEAU_TORNADO}")

2026-03-27 18:21:48,552 - INFO - tableau_tornado : (59, 14)  → /Users/fabianfuentes/git/ssp_uganda_data/ssp_modeling/Tableau/data/tableau_tornado.csv


## 6 · Summary

In [7]:
import pandas as pd

summary = pd.DataFrame([
    {"output": "decomposed SSP",    "rows": len(df_decomposed), "cols": df_decomposed.shape[1], "file": cfg.OUTPUT_DECOMPOSED.name},
    {"output": "cost-benefits data","rows": len(cb_data),       "cols": cb_data.shape[1],       "file": cfg.OUTPUT_CB_DATA.name},
    {"output": "MAC curves",        "rows": len(mac_df),        "cols": mac_df.shape[1],        "file": cfg.OUTPUT_MAC.name},
])
print(summary.to_string(index=False))

            output   rows  cols                                 file
    decomposed SSP   3068  4054    decomposed_ssp_output_tornado.csv
cost-benefits data 505820    21       cost_benefits_data_tornado.csv
        MAC curves     44     9 marginal_abatement_costs_tornado.csv
